# Unconstrained optimization

_______

## 1) Function definition

We study :
- Ellipsoid : $f(x,y) \mapsto (x-1)^2 + 100(y-1)^2$
- Rosenbrock's banana : $f(x,y) \mapsto (x-1)^2 + 100(y-x^2)^2$

In both cases, the optimum point is $(x,y)^* = (1,1)$

In [ ]:
rosenbrock = True
ellipsoid = not rosenbrock

import numpy as np

if ellipsoid: 
    f = lambda x, y: (x - 1)**2 + 100 * (y - 1)**2
    df = lambda x, y: np.array([
        2 * (x - 1),
        200 * (y - 1)
    ])
    d2f = lambda x, y: np.array([
        [2, 0],
        [0, 200]
    ])

elif rosenbrock:  # Rosenbrock
    f = lambda x, y: (1 - x)**2 + 100 * (y - x**2)**2
    df = lambda x, y: np.array([
        2 * x - 400 * x * (y - x**2) - 2,
        200 * (y - x**2)
    ])
    d2f = lambda x, y: np.array([
        [1200 * x**2 - 400 * y + 2, -400 * x],
        [-400 * x, 200]
    ])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plotf(f):
    x, y = np.meshgrid( np.arange(-5, 5.05, 0.05), np.arange(-5, 5.05, 0.05))
    ff = f(x, y)
    plt.contour(x, y, np.log(ff + 1), levels=10)
    plt.gca().set_aspect('auto')
    # Optimum
    plt.scatter( 1, 1, color='r', marker='o', linewidth=2)
    plt.grid(True)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Function isolines and optimum")

plotf(f)
plt.show()

## 1) Fixed step descent

In [ ]:
from numpy.linalg import norm

alpha = 0.001  # fixed step
maxit = 100

# stop criterion
atol = 1e-6
rtol = 1e-6

# initialization
x_fixed_step = [-2.0]
y_fixed_step = [-0.5]

obj_fixed_step = [f(x_fixed_step[0], y_fixed_step[0])]

# loop
for i in range(maxit):
    # compute gradient
    grad = df(x_fixed_step[i], y_fixed_step[i])
    # update
    x_fixed_step.append(x_fixed_step[i] - alpha * grad[0])
    y_fixed_step.append(y_fixed_step[i] - alpha * grad[1])
    obj_fixed_step.append(f(x_fixed_step[-1], y_fixed_step[-1]))
    
    # display
    print(f"it {i+1} | obj = {obj_fixed_step[-1]:.6e} | |dF| = {norm(grad):.6e}")

    # stopping criteria
    if norm(grad) < atol or norm(grad) / norm(df(x_fixed_step[0], y_fixed_step[0])) < rtol :
        print("Converged!")
        break

In [ ]:
plotf(f)
plt.plot(x_fixed_step, y_fixed_step, '*-')
plt.title(f"Fixed-step gradient descent (step size = {alpha})")
plt.show()

___
## 2) Adaptative step size
### a) Line search (Armijo)

In [ ]:
def armijo(f, x0, y0, df0, s = 0.1, coeff_step = 0.5, step = 1e3):
    """ Armijo backtracking """
    maxit = 100
    i = 0

    while f(x0 - step * df0[0], y0 - step * df0[1]) >= f(x0, y0) - step * s * np.dot(df0, df0) and i < maxit:
        step *= coeff_step
        i += 1
    return step

#### Optimization loop


In [ ]:
s = 0.1  # tolerance
coeff_step = 0.5

x_line_search = [-2.0]
y_line_search = [-0.5]

obj_line_search = [f(x_line_search[0], y_line_search[0])]

for i in range(maxit):

    # compute gradient
    grad = df(x_line_search[i], y_line_search[i])

    # line search
    alpha = armijo(f, x_line_search[i], y_line_search[i], grad, s, coeff_step)

    # update
    x_line_search.append(x_line_search[i] - alpha * grad[0])
    y_line_search.append(y_line_search[i] - alpha * grad[1])

    obj_line_search.append( f(x_line_search[-1], y_line_search[-1]) )
    
    # display
    print(f"it {i+1} | obj = {obj_line_search[-1]:.6e} | |dF| = {norm(grad):.6e}")

    # stopping criteria
    if norm(grad) < atol  or   norm(grad) / norm( df(x_line_search[0], y_line_search[0])) < rtol:
        print("Converged!")
        break

In [ ]:
plotf(f)
plt.plot(x_line_search, y_line_search, '*-')
plt.title(f"Gradient descent with Armijo's backtracking")
plt.show()

___
### b) Optimal step size (steepest descent)

In [ ]:
x_steepest_descent = [-2.0]
y_steepest_descent = [-0.5]

obj_steepest_descent = [f(x_steepest_descent[0], y_steepest_descent[0])]

for i in range(maxit):

    # gradient
    grad = df(x_steepest_descent[i], y_steepest_descent[i])

    # exhaustive (but costly) line search
    alpha = armijo(f,x_steepest_descent[i],y_steepest_descent[i],grad,0,0.7)
    alphaTest = np.linspace(0, 2 * alpha, 100)
    values = f( x_steepest_descent[i] - alphaTest * grad[0], y_steepest_descent[i] - alphaTest * grad[1])
    iAlpha = np.argmin(values)
    alpha = alphaTest[iAlpha]

    # update
    x_steepest_descent.append(x_steepest_descent[i] - alpha * grad[0])
    y_steepest_descent.append(y_steepest_descent[i] - alpha * grad[1])
    obj_steepest_descent.append(f(x_steepest_descent[-1], y_steepest_descent[-1]))

    # show
    print(f"it {i+1} | obj = {obj_steepest_descent[-1]:.6e} | |dF| = {norm(grad):.6e}")
    
    # stop criteria
    if norm(grad) < atol or norm(grad) / norm( df(x_steepest_descent[0], y_steepest_descent[0])) < rtol:
        print("Converged!")
        break

    

In [ ]:
plotf(f)
plt.plot(x_steepest_descent, y_steepest_descent, '*-')
plt.title(f"Steepest gradient descent")
plt.show()

____
### c) Preconditioned gradient descent

In [ ]:
s = 0.5
coeff = 0.5

x_pc = [-2.0]
y_pc = [-0.5]

obj_pc = [f(x_pc[0], y_pc[0])]

for i in range(maxit):

    # gradient
    grad = df(x_pc[i], y_pc[i])

    # preconditioning
    P = np.array([ [i % 2, 0], 
                   [0, (i + 1) % 2]])
    d = -P @ grad

    # Armijo line search
    alpha = armijo( f, x_pc[i], y_pc[i], -d, s, coeff)

    # update
    x_pc.append( x_pc[i] + alpha * d[0])
    y_pc.append(y_pc[i] + alpha * d[1])
    obj_pc.append(f(x_pc[-1], y_pc[-1]))
    
    # display
    print( f"it {i+1} | obj = {obj_pc[-1]:.6e} | |dF| = {norm(grad):.6e}")

    # stop criteria
    if norm(grad) < atol or norm(grad) / norm(df(x_pc[0], y_pc[0])) < rtol :
        print("Converged!")
        break


In [ ]:
plotf(f)
plt.plot(x_pc, y_pc, '*-')
plt.title(f"Preconditioned (coordinate) gradient descent")
plt.show()

____

## 2) Newton

### a) No step control

In [ ]:
x_newton = [-2.0]
y_newton = [-0.5]

obj_newton = [f(x_newton[0], y_newton[0])]

for i in range(maxit):

    # df/dx and d²f/dx²
    grad = df(x_newton[i], y_newton[i])
    hess = d2f(x_newton[i], y_newton[i])

    # descent direction
    d = -np.linalg.solve(hess, grad)

    # update
    x_newton.append( x_newton[i] + d[0])
    y_newton.append( y_newton[i] + d[1])
    obj_newton.append(f(x_newton[-1], y_newton[-1]))
    
    # display
    print( f"it {i+1} | obj = {obj_newton[-1]:.6e} | |dF| = {norm(grad):.6e}")

    # stop criterion
    if norm(grad) < atol  or norm(grad) / norm(df(x_newton[0], y_newton[0]) ) < rtol :
        print("Converged!")
        break

In [ ]:
plotf(f)
plt.plot(x_newton, y_newton, '*-')
plt.title(f"Newton descent - unit step")
plt.show()

____
### b) With line-search

In [ ]:
x_newton_ls = [-2.0]
y_newton_ls = [-0.5]

obj_newton_ls = [f(x_newton_ls[0], y_newton_ls[0])]

for i in range(maxit):

    # df/dx and d²f/dx²
    grad = df(x_newton_ls[i], y_newton_ls[i])
    hess = d2f(x_newton_ls[i], y_newton_ls[i])

    # descent direction
    d = -np.linalg.solve(hess, grad)

    # line search
    alpha = armijo( f, x_newton_ls[i], y_newton_ls[i], -d, 0, 0.7, step = 1)

    # update
    x_newton_ls.append( x_newton_ls[i] + alpha * d[0])
    y_newton_ls.append( y_newton_ls[i] + alpha * d[1])

    obj_newton_ls.append(f(x_newton_ls[-1], y_newton_ls[-1]))
    
    # display
    print( f"it {i+1} | obj = {obj_newton_ls[-1]:.6e} | |dF| = {norm(grad):.6e}")

    # stop criteria
    if norm(grad) < atol  or norm(grad) / norm(df(x_newton_ls[0], y_newton_ls[0]) ) < rtol :
        print("Converged!")
        break

In [ ]:
plotf(f)
plt.plot(x_newton_ls, y_newton_ls, '*-')
plt.title(f"Newton descent - line search")
plt.show()